# Credit Scoring — Exploratory Data Analysis

**Goal**: Understand the Home Credit Default Risk dataset, identify key predictive variables, and prepare the feature engineering pipeline to predict the probability of loan default (`TARGET = 1`).

**Dataset**: Home Credit Default Risk — [Kaggle Competition](https://www.kaggle.com/c/home-credit-default-risk)

**Metric**: AUC-ROC (target: ≥ 55%, ideally ≥ 62%)

---

## Table of Contents
1. [Imports & Setup](#1)
2. [Load Data](#2)
3. [Target Variable Distribution](#3)
4. [Missing Values](#4)
5. [Variable Types](#5)
6. [Numerical Features Analysis](#6)
7. [Categorical Features Analysis](#7)
8. [External Score Features (EXT_SOURCE)](#8)
9. [Correlation with TARGET](#9)
10. [Anomaly Detection](#10)
11. [Overview of Supplementary Tables](#11)
12. [Conclusions & Feature Engineering Plan](#12)

<a id='1'></a>
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

%matplotlib inline

DATA_PATH = '../../data/'
print('Setup complete.')

<a id='2'></a>
## 2. Load Data

We start with the two main tables: `application_train.csv` (labelled) and `application_test.csv` (unlabelled — Kaggle submission).

In [ ]:
train = pd.read_csv(DATA_PATH + 'application_train.csv')
test  = pd.read_csv(DATA_PATH + 'application_test.csv')

print(f'Train : {train.shape[0]:,} rows  × {train.shape[1]} columns')
print(f'Test  : {test.shape[0]:,} rows  × {test.shape[1]} columns')
print(f'\nTrain columns not in Test : {set(train.columns) - set(test.columns)}')

In [ ]:
# Quick overview of the main table
train.head(3)

In [ ]:
train.info(verbose=False, show_counts=True)

<a id='3'></a>
## 3. Target Variable Distribution

The target column `TARGET` is binary:
- **0** = client repaid the loan (no default)
- **1** = client defaulted (payment difficulties)

Understanding class imbalance is crucial because it directly affects the choice of evaluation metric.

In [ ]:
counts = train['TARGET'].value_counts()
pcts   = train['TARGET'].value_counts(normalize=True) * 100

print('TARGET distribution:')
print(f'  0 (no default) : {counts[0]:,}  ({pcts[0]:.2f}%)')
print(f'  1 (default)    : {counts[1]:,}  ({pcts[1]:.2f}%)')
print(f'  Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(['No Default (0)', 'Default (1)'], counts.values,
                   color=['#27ae60', '#e74c3c'], edgecolor='black', linewidth=0.8)
for bar, pct in zip(bars, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{pct:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Target Variable Distribution', fontsize=13)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=['No Default', 'Default'],
            colors=['#27ae60', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Balance', fontsize=13)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('=> Strong class imbalance (~92% vs ~8%).')
print('=> ACCURACY is misleading here: a model that always predicts 0 gets 92% accuracy.')
print('=> We use AUC-ROC as the evaluation metric.')

<a id='4'></a>
## 4. Missing Values Analysis

In [ ]:
missing      = train.isnull().sum()
missing_pct  = (missing / len(train) * 100).round(2)
missing_df   = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df   = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)

print(f'Columns with missing values   : {len(missing_df)} / {train.shape[1]}')
print(f'Columns with >50% missing     : {(missing_df["pct"] > 50).sum()}')
print(f'Columns with >80% missing     : {(missing_df["pct"] > 80).sum()}')
print('\nTop 25 columns with most missing values:')
missing_df.head(25)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
top30 = missing_df.head(30)
colors = ['#e74c3c' if p > 50 else '#e67e22' if p > 20 else '#3498db'
          for p in top30['pct']]
ax.barh(top30.index, top30['pct'], color=colors)
ax.axvline(50, color='red',    linestyle='--', linewidth=1.5, label='50% threshold')
ax.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='20% threshold')
ax.set_xlabel('Missing value percentage (%)')
ax.set_title('Top 30 columns — missing values', fontsize=13)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nStrategy:')
print('  >50% missing  -> consider dropping or imputing with special "UNKNOWN" category')
print('  Numerical     -> impute with median')
print('  Categorical   -> impute with mode or "MISSING"')

<a id='5'></a>
## 5. Variable Types

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['TARGET', 'SK_ID_CURR']]

print(f'Numerical features   : {len(num_cols)}')
print(f'Categorical features : {len(cat_cols)}')
print(f'\nCategorical features:')
for col in cat_cols:
    n_unique = train[col].nunique()
    print(f'  {col:40s} — {n_unique} unique values')

<a id='6'></a>
## 6. Numerical Features Analysis

We focus on the most interpretable and likely predictive numerical features.

In [ ]:
# Create derived features for readability
train['AGE_YEARS']         = (-train['DAYS_BIRTH']) / 365
train['YEARS_EMPLOYED']    = train['DAYS_EMPLOYED'].replace(365243, np.nan) / -365
train['CREDIT_INCOME_RATIO']  = train['AMT_CREDIT'] / train['AMT_INCOME_TOTAL']
train['ANNUITY_INCOME_RATIO'] = train['AMT_ANNUITY'] / train['AMT_INCOME_TOTAL']

key_features = {
    'AGE_YEARS':            'Age (years)',
    'YEARS_EMPLOYED':       'Years employed',
    'AMT_INCOME_TOTAL':     'Annual income',
    'AMT_CREDIT':           'Credit amount',
    'AMT_ANNUITY':          'Loan annuity',
    'CREDIT_INCOME_RATIO':  'Credit / Income ratio',
    'ANNUITY_INCOME_RATIO': 'Annuity / Income ratio',
    'CNT_CHILDREN':         'Number of children',
}

fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, (col, label) in enumerate(key_features.items()):
    ax = axes[i]
    for target_val, color, lbl in [(0, '#27ae60', 'No Default'), (1, '#e74c3c', 'Default')]:
        data = train[train['TARGET'] == target_val][col].dropna()
        # Cap extreme values for readability
        p1, p99 = data.quantile(0.01), data.quantile(0.99)
        data = data.clip(p1, p99)
        ax.hist(data, bins=50, alpha=0.55, color=color, label=lbl, density=True)
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.suptitle('Distribution of Key Features by Target', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mean of key features by TARGET
features_to_compare = list(key_features.keys())
comparison = train.groupby('TARGET')[features_to_compare].mean().T
comparison.columns = ['No Default (0)', 'Default (1)']
comparison['Difference (%)'] = ((comparison['Default (1)'] - comparison['No Default (0)'])
                                 / comparison['No Default (0)'].abs() * 100).round(1)
comparison

<a id='7'></a>
## 7. Categorical Features Analysis

We compute the **default rate** for each category to identify high-risk groups.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(cat_cols[:6]):
    ax = axes[i]
    default_rate = (train.groupby(col)['TARGET']
                    .agg(['mean', 'count'])
                    .rename(columns={'mean': 'default_rate', 'count': 'n'})
                    .sort_values('default_rate', ascending=False))
    
    bars = ax.bar(range(len(default_rate)), default_rate['default_rate'],
                  color='#3498db', edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(default_rate)))
    ax.set_xticklabels(default_rate.index, rotation=35, ha='right', fontsize=8)
    ax.set_title(f'Default rate by {col}', fontsize=10)
    ax.set_ylabel('Default rate')
    ax.axhline(train['TARGET'].mean(), color='red', linestyle='--',
               linewidth=1, label=f'Overall: {train["TARGET"].mean():.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Default Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print default rates for all categorical variables
overall_rate = train['TARGET'].mean()
print(f'Overall default rate: {overall_rate:.4f} ({overall_rate*100:.2f}%)\n')

for col in cat_cols:
    rates = (train.groupby(col)['TARGET'].mean()
             .sort_values(ascending=False)
             .to_frame('default_rate'))
    rates['vs_average'] = rates['default_rate'] - overall_rate
    print(f'--- {col} ---')
    print(rates.to_string())
    print()

<a id='8'></a>
## 8. External Score Features (EXT_SOURCE)

`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3` are normalized scores from external data sources.
They are among the **strongest predictors** of default in this dataset.

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

print('Correlation with TARGET:')
for col in ext_cols:
    corr = train[col].corr(train['TARGET'])
    missing_pct_col = train[col].isnull().mean() * 100
    print(f'  {col}: r = {corr:.4f}  |  {missing_pct_col:.1f}% missing')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ext_cols):
    for target_val, color, lbl in [(0, '#27ae60', 'No Default'), (1, '#e74c3c', 'Default')]:
        data = train[train['TARGET'] == target_val][col].dropna()
        ax.hist(data, bins=50, alpha=0.6, color=color, label=lbl, density=True)
    corr = train[col].corr(train['TARGET'])
    ax.set_title(f'{col}\nr = {corr:.4f} with TARGET', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('Score value')
    ax.set_ylabel('Density')

plt.suptitle('EXT_SOURCE Scores Distribution by Target', fontsize=13)
plt.tight_layout()
plt.savefig('ext_source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Combined EXT_SOURCE feature
train['EXT_SOURCE_MEAN']   = train[ext_cols].mean(axis=1)
train['EXT_SOURCE_MIN']    = train[ext_cols].min(axis=1)
train['EXT_SOURCE_PROD']   = train[ext_cols].prod(axis=1)

print('Correlation of combined EXT_SOURCE features with TARGET:')
for feat in ['EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_PROD']:
    r = train[feat].corr(train['TARGET'])
    print(f'  {feat}: r = {r:.4f}')

<a id='9'></a>
## 9. Correlation with TARGET — Top Predictive Features

In [ ]:
# Select only numeric columns (excluding IDs)
numeric_df    = train.select_dtypes(include=[np.number]).drop(columns=['SK_ID_CURR'])
correlations  = numeric_df.corr()['TARGET'].drop('TARGET')
corr_abs      = correlations.abs().sort_values(ascending=False)

print('Top 25 features by absolute correlation with TARGET:')
top25 = corr_abs.head(25)
df_corr = pd.DataFrame({
    'correlation': correlations[top25.index],
    'abs_correlation': top25
})
print(df_corr.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
top20 = correlations[corr_abs.head(20).index].sort_values()
colors = ['#e74c3c' if v > 0 else '#27ae60' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Top 20 Features — Pearson Correlation with TARGET', fontsize=12)
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.savefig('correlations_with_target.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPositive correlation (red)  -> feature increases default probability')
print('Negative correlation (green)-> feature decreases default probability')

<a id='10'></a>
## 10. Anomaly Detection

Some features contain anomalous values that must be handled before training.

In [ ]:
# DAYS_EMPLOYED anomaly: value 365243 represents retired/unemployed clients
print('DAYS_EMPLOYED statistics:')
print(train['DAYS_EMPLOYED'].describe())

n_anom = (train['DAYS_EMPLOYED'] == 365243).sum()
print(f'\nAnomalous value 365243: {n_anom:,} rows ({n_anom/len(train)*100:.1f}%)')

rate_anom   = train[train['DAYS_EMPLOYED'] == 365243]['TARGET'].mean()
rate_normal = train[train['DAYS_EMPLOYED'] != 365243]['TARGET'].mean()
print(f'Default rate — anomalous rows : {rate_anom:.4f}')
print(f'Default rate — normal rows    : {rate_normal:.4f}')
print('=> The anomalous rows carry information — create a binary flag.')

In [ ]:
# Check other potential anomalies
print('AMT_INCOME_TOTAL — top 10 values:')
print(train['AMT_INCOME_TOTAL'].nlargest(10).values)

print('\nDAYS_BIRTH range (in years):')
print(f'  Min age: {-train["DAYS_BIRTH"].max()/365:.1f} years')
print(f'  Max age: {-train["DAYS_BIRTH"].min()/365:.1f} years')

print('\nCNT_CHILDREN distribution:')
print(train['CNT_CHILDREN'].value_counts().head(10))

<a id='11'></a>
## 11. Overview of Supplementary Tables

These tables provide **historical behavioral data** — very useful for feature engineering but need aggregation before joining to the main table.

In [ ]:
import os

supplementary_files = {
    'bureau.csv':                'Previous credits at other financial institutions',
    'bureau_balance.csv':        'Monthly balance of bureau credits',
    'previous_application.csv':  'Previous Home Credit loan applications',
    'POS_CASH_balance.csv':      'Monthly POS/cash loan snapshots',
    'credit_card_balance.csv':   'Monthly credit card balance snapshots',
    'installments_payments.csv': 'Installment payment history',
}

print(f'{"File":<35} {"Description":<45} {"Size (MB)":>10}')
print('-' * 95)
for fname, desc in supplementary_files.items():
    fpath = DATA_PATH + fname
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{fname:<35} {desc:<45} {size_mb:>10.1f}')

In [ ]:
# Quick peek at bureau.csv
bureau = pd.read_csv(DATA_PATH + 'bureau.csv', nrows=5000)
print(f'bureau.csv columns ({len(bureau.columns)}):')
print(list(bureau.columns))
print(f'\nRows per SK_ID_CURR (mean): {bureau.groupby("SK_ID_CURR").size().mean():.2f}')
print(f'Rows per SK_ID_CURR (max) : {bureau.groupby("SK_ID_CURR").size().max()}')

In [ ]:
# Quick peek at previous_application.csv
prev = pd.read_csv(DATA_PATH + 'previous_application.csv', nrows=5000)
print(f'previous_application.csv columns ({len(prev.columns)}):')
print(list(prev.columns))
print(f'\nContract status distribution:')
if 'NAME_CONTRACT_STATUS' in prev.columns:
    print(prev['NAME_CONTRACT_STATUS'].value_counts().to_string())

del prev, bureau  # Free memory

<a id='12'></a>
## 12. Conclusions & Feature Engineering Plan

### Key Findings

| Finding | Impact |
|---|---|
| Strong class imbalance (~92% vs ~8%) | Use AUC-ROC, not accuracy; use `class_weight='balanced'` |
| `EXT_SOURCE_1/2/3` are top predictors | Always include; handle ~56% missing in EXT_SOURCE_1 |
| `DAYS_EMPLOYED = 365243` anomaly | Create `DAYS_EMPLOYED_ANOM` binary flag |
| Multiple supplementary tables | Aggregation features = key to AUC > 62% |

In [ ]:
print("""
=============================================================
   FEATURE ENGINEERING PLAN
=============================================================

--- From application_train/test.csv ---

  New features:
  - AGE_YEARS                  = -DAYS_BIRTH / 365
  - DAYS_EMPLOYED_ANOM         = 1 if DAYS_EMPLOYED == 365243 else 0
  - DAYS_EMPLOYED (clean)      = replace 365243 with NaN
  - CREDIT_INCOME_RATIO        = AMT_CREDIT / AMT_INCOME_TOTAL
  - ANNUITY_INCOME_RATIO       = AMT_ANNUITY / AMT_INCOME_TOTAL
  - CREDIT_GOODS_RATIO         = AMT_CREDIT / AMT_GOODS_PRICE
  - EXT_SOURCE_MEAN            = mean(EXT_SOURCE_1, 2, 3)
  - EXT_SOURCE_MIN             = min(EXT_SOURCE_1, 2, 3)
  - INCOME_PER_PERSON          = AMT_INCOME_TOTAL / CNT_FAM_MEMBERS

--- From bureau.csv (aggregate per SK_ID_CURR) ---

  - BUREAU_COUNT               = number of past credits
  - BUREAU_ACTIVE_COUNT        = number of active credits
  - BUREAU_DAYS_CREDIT_MEAN    = avg days since past credit applied
  - BUREAU_CREDIT_SUM_DEBT     = total debt in bureau credits
  - BUREAU_OVERDUE_SUM         = total overdue amounts

--- From previous_application.csv (aggregate per SK_ID_CURR) ---

  - PREV_COUNT                 = number of previous applications
  - PREV_APPROVED_COUNT        = number approved
  - PREV_REFUSED_COUNT         = number refused
  - PREV_APPROVE_RATE          = approved / total
  - PREV_AMT_CREDIT_MEAN       = avg credit amount of past loans

--- From installments_payments.csv (aggregate per SK_ID_CURR) ---

  - INSTAL_PAYMENT_DIFF_MEAN   = avg(amt_payment - amt_instalment)
  - INSTAL_DAYS_LATE_MEAN      = avg days past due
  - INSTAL_LATE_COUNT          = number of late payments

--- Model ---

  Baseline : Logistic Regression (interpretable, fast)
  Main     : LightGBM (handles missing values natively, best AUC)
  Tuning   : Optuna or GridSearchCV

=============================================================
""")